# Notebook 09 — Mini-Project: PCA on a Public Expression Dataset

**Module:** 04 — Linear Algebra  
**Notebook:** 09 of 10 — Portfolio artifact  
**Prerequisites:** NB05, NB06  
**Time estimate:** 3–4 hours (spread over 2 sessions)

> **Portfolio artifact.** This notebook produces a publication-quality PCA analysis
> figure saved to `portfolio/module04_pca_expression_analysis.png`.
> Every cell must run from a clean environment. Data is fetched programmatically
> — never committed to the repository.

---
## 1. Project Statement

**Task:** Apply PCA from scratch to a real public expression dataset. Produce a
4-panel portfolio figure with full biological interpretation.

**Dataset options (choose one):**
- Option A: GEO accession GSE2034 (used in Module 03) — breast cancer, 286 samples
- Option B: Any GEO series with ≥30 samples and ≥2 biological groups
- Option C: Synthetic fallback (Cell 3 below) — ensures reproducibility without download

**Pre-specified analysis plan:**
1. Load and normalize data
2. Filter: top 5,000 most-variable genes
3. Center
4. PCA from scratch (`np.linalg.svd`) — no sklearn for the main analysis
5. Produce 4-panel figure: scree + PC1×PC2 scatter + loadings + heatmap
6. Validate against sklearn
7. Write 3-sentence biological interpretation

In [ ]:
# Cell 1 — Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.decomposition import PCA as sklearn_PCA

In [ ]:
# Cell 2 — PCA from scratch (using SVD — production-quality)
def pca_svd(X: np.ndarray, n_components: int) -> dict:
    """
    PCA via SVD of the centered data matrix.
    Preferred over eigendecomposition when n_features >> n_samples.

    Parameters
    ----------
    X : (n_samples, n_features) — centered internally

    Returns
    -------
    dict: scores, loadings, singular_values, explained_variance_ratio
    """
    X = np.asarray(X, dtype=float)
    n_samples = X.shape[0]
    X_centered = X - X.mean(axis=0)
    # full_matrices=False: compact SVD, shapes (n,k), (k,), (k,p)
    U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)
    # PC scores: U * s (= X_centered @ Vt.T for top k)
    scores   = U[:, :n_components] * s[:n_components]
    loadings = Vt[:n_components].T   # (n_features, n_components)
    explained_variance_ratio = (s**2 / (s**2).sum())[:n_components]
    return dict(scores=scores, loadings=loadings,
                singular_values=s,
                explained_variance_ratio=explained_variance_ratio,
                X_centered=X_centered)

In [ ]:
# Cell 3 — Synthetic fallback dataset
# Simulates a realistic bulk RNA-seq experiment: 3 tissue types, 286 samples
# Replace this cell with real GEO data loading if available

rng = np.random.default_rng(2034)

N_GENES   = 22_283
N_SAMPLES = 286
N_TISSUE  = 3
TISSUE_SIZES = [100, 100, 86]

# Simulate expression as log2-expression values (Affymetrix-like)
base_expr = rng.normal(8.0, 2.5, N_GENES)
tissue_effects = rng.normal(0, 3.0, (N_TISSUE, N_GENES))
tissue_labels  = np.repeat(np.arange(N_TISSUE), TISSUE_SIZES)

X_raw = (base_expr[None, :]
         + tissue_effects[tissue_labels]
         + rng.normal(0, 0.8, (N_SAMPLES, N_GENES)))

sample_names = [f"T{t}_S{i}" for t, size in enumerate(TISSUE_SIZES)
                for i in range(size)]
tissue_names = ['Liver', 'Brain', 'Muscle']

print(f"Expression matrix: {X_raw.shape}  (samples × genes)")
print(f"Sample groups: {[f'{n}({s})' for n, s in zip(tissue_names, TISSUE_SIZES)]}")

In [ ]:
# Cell 4 — Filter: top 5,000 most-variable genes
gene_variances = X_raw.var(axis=0)
top5k_idx = np.argsort(gene_variances)[-5000:]
X_filtered = X_raw[:, top5k_idx]
print(f"After variance filter: {X_filtered.shape}")

In [ ]:
# Cell 5 — PCA from scratch
n_pcs = 20
pca_result = pca_svd(X_filtered, n_components=n_pcs)

print(f"PC scores shape:    {pca_result['scores'].shape}")
print(f"Loadings shape:     {pca_result['loadings'].shape}")
print(f"Explained variance (top 5): {pca_result['explained_variance_ratio'][:5].round(4)}")
print(f"Cumulative (top 3): {pca_result['explained_variance_ratio'][:3].sum():.2%}")

# Validate against sklearn
sk = sklearn_PCA(n_components=n_pcs, random_state=42).fit(X_filtered)
evr_match = np.allclose(pca_result['explained_variance_ratio'],
                         sk.explained_variance_ratio_, atol=1e-6)
print(f"\nValidation vs sklearn (explained var): {evr_match}")

In [ ]:
# Cell 6 — Portfolio figure: 4-panel PCA analysis
evr = pca_result['explained_variance_ratio']
scores = pca_result['scores']
loadings = pca_result['loadings']

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.40, wspace=0.35)

tissue_colors = ['steelblue', 'tomato', 'green']

# Panel A: Scree plot
ax = fig.add_subplot(gs[0, 0])
ax.bar(range(1, n_pcs+1), evr, color='steelblue', alpha=0.8, label='Per-PC variance')
ax.plot(range(1, n_pcs+1), np.cumsum(evr), 'tomato', lw=2, marker='o', ms=4, label='Cumulative')
ax.axhline(0.9, color='gray', ls='--', lw=1, label='90%')
ax.set_xlabel('Principal component', fontsize=10)
ax.set_ylabel('Explained variance ratio', fontsize=10)
ax.set_title('A. Scree plot', fontsize=11)
ax.legend(fontsize=8)

# Panel B: PC1 × PC2 scatter
ax = fig.add_subplot(gs[0, 1])
for t, (name, color) in enumerate(zip(tissue_names, tissue_colors)):
    mask = tissue_labels == t
    ax.scatter(scores[mask, 0], scores[mask, 1], s=20, alpha=0.8,
               color=color, label=name)
ax.set_xlabel(f'PC1 ({evr[0]:.1%})', fontsize=10)
ax.set_ylabel(f'PC2 ({evr[1]:.1%})', fontsize=10)
ax.set_title('B. PC1 × PC2 — tissue types separate', fontsize=11)
ax.legend(fontsize=9)

# Panel C: Top 20 gene loadings on PC1 and PC2 (biplot-style)
ax = fig.add_subplot(gs[1, 0])
top20_pc1 = np.argsort(np.abs(loadings[:, 0]))[-20:]
y_pos = np.arange(20)
ax.barh(y_pos - 0.2, loadings[top20_pc1, 0], height=0.35,
        color='steelblue', alpha=0.8, label='PC1')
ax.barh(y_pos + 0.2, loadings[top20_pc1, 1], height=0.35,
        color='tomato', alpha=0.8, label='PC2')
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Loading coefficient', fontsize=10)
ax.set_ylabel('Gene rank', fontsize=10)
ax.set_title('C. Top 20 gene loadings (PC1 & PC2)', fontsize=11)
ax.legend(fontsize=8)

# Panel D: Heatmap of PC scores (top 5 PCs × all samples)
ax = fig.add_subplot(gs[1, 1])
score_mat = scores[:, :5].T    # (5, n_samples)
order = np.argsort(tissue_labels)
im = ax.imshow(score_mat[:, order], aspect='auto', cmap='RdBu_r',
               vmin=-30, vmax=30, interpolation='none')
plt.colorbar(im, ax=ax, fraction=0.03, label='PC score')
cum_sizes = np.cumsum(TISSUE_SIZES[:-1])
for b in cum_sizes:
    ax.axvline(b - 0.5, color='yellow', lw=1)
ax.set_xticks([])
ax.set_yticks(range(5))
ax.set_yticklabels([f'PC{i+1} ({evr[i]:.1%})' for i in range(5)], fontsize=8)
ax.set_xlabel('Samples (sorted by tissue)', fontsize=10)
ax.set_title('D. Top 5 PC scores across samples', fontsize=11)

fig.suptitle(
    'Module 04 — PCA of Expression Data\n'
    f'n={N_SAMPLES} samples × top 5,000 variable genes | SVD-based PCA from scratch',
    fontsize=12, fontweight='bold'
)

portfolio_path = Path('../../../portfolio/module04_pca_expression_analysis.png')
portfolio_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(portfolio_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Portfolio figure saved → {portfolio_path.resolve()}")

---
## Biological Interpretation

**Write 3 sentences here after running the analysis:**

> *PC1 explains ____% of variance and separates _____ from the other tissues,
> suggesting that the dominant source of transcriptional variation is _____.
> PC2 explains ____% and further resolves _____ from _____, likely reflecting _____.
> The gene loadings on PC1 include _____, consistent with tissue-specific gene expression.*

---
## Reflection

**Date completed:** ____________________

> *[Did the tissue types separate cleanly on PC1/PC2? What would you do next — UMAP, clustering, or gene ontology enrichment of the top loadings?]*

---
*Next: `10_module_assessment.ipynb`*